# Pertemuan 11 - Unsupervised Learning: Clustering

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Mata Kuliah:** Pengantar Data Science  
**Topik:** K-Means, Elbow Method, Silhouette Score, dan Hierarchical Clustering

Notebook ini dibuat untuk praktikum Pertemuan 11. Studi kasus yang digunakan adalah **segmentasi pelanggan** berdasarkan pendapatan tahunan dan skor belanja. Karena clustering termasuk **Unsupervised Learning**, data tidak memiliki label target seperti pada klasifikasi.


## 1. Import Library

Library yang digunakan adalah Pandas dan NumPy untuk pengolahan data, Matplotlib dan Seaborn untuk visualisasi, Scikit-Learn untuk K-Means dan scaling, serta SciPy untuk dendrogram Hierarchical Clustering.


In [ ]:
# Jalankan notebook ini di Google Colab atau Jupyter Notebook.
# Jika ada library yang belum tersedia, hapus tanda pagar pada baris berikut:
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


## 2. Generate dan Eksplorasi Dataset

Dataset dibuat secara sintetis dengan tiga kelompok tersembunyi:

- Kelompok hemat: pendapatan rendah dan skor belanja rendah
- Kelompok menengah: pendapatan sedang dan skor belanja sedang
- Kelompok premium: pendapatan tinggi dan skor belanja tinggi

Kolom tambahan `usia` dan `gender` ditambahkan agar dataset terlihat lebih realistis, tetapi fokus clustering tetap memakai `pendapatan_tahunan` dan `skor_belanja`.


In [ ]:
np.random.seed(42)

grp1 = np.random.normal([30, 20], [6, 8], (100, 2))    # hemat
grp2 = np.random.normal([70, 55], [8, 10], (100, 2))   # menengah
grp3 = np.random.normal([110, 85], [10, 8], (100, 2))  # premium

data = np.vstack([grp1, grp2, grp3])

df = pd.DataFrame(data, columns=["pendapatan_tahunan", "skor_belanja"])
df["usia"] = np.random.randint(18, 65, len(df))
df["gender"] = np.random.choice(["L", "P"], len(df))

# Membatasi nilai agar tetap masuk akal.
df["pendapatan_tahunan"] = df["pendapatan_tahunan"].clip(15, 150)
df["skor_belanja"] = df["skor_belanja"].clip(1, 100)

print("Shape dataset:", df.shape)
display(df.head())
display(df.describe().round(2))


In [ ]:
print("Distribusi gender:")
display(df["gender"].value_counts())

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="pendapatan_tahunan",
    y="skor_belanja",
    hue="gender",
    alpha=0.7,
)
plt.title("Sebaran Pendapatan Tahunan dan Skor Belanja")
plt.xlabel("Pendapatan Tahunan (juta Rp)")
plt.ylabel("Skor Belanja")
plt.show()


**Interpretasi EDA:** Dari scatter plot awal, terlihat ada kecenderungan beberapa kelompok pelanggan berdasarkan pendapatan tahunan dan skor belanja. Karena belum ada label kategori pelanggan, teknik clustering digunakan untuk menemukan kelompok tersebut secara otomatis.


## 3. Preprocessing Data

Fitur numerik yang dipakai untuk clustering adalah `pendapatan_tahunan` dan `skor_belanja`. Kedua fitur ini distandarisasi menggunakan `StandardScaler` agar skala nilainya seimbang saat dihitung jaraknya oleh K-Means.


In [ ]:
features = ["pendapatan_tahunan", "skor_belanja"]
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Rata-rata setelah scaling:", X_scaled.mean(axis=0).round(3))
print("Std setelah scaling      :", X_scaled.std(axis=0).round(3))


## 4. Elbow Method untuk Menentukan K

Metode Elbow digunakan untuk melihat nilai WCSS/Inertia dari beberapa pilihan jumlah cluster. Nilai K yang baik biasanya berada pada titik ketika penurunan WCSS mulai melandai.


In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, init="k-means++", n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

elbow_df = pd.DataFrame({
    "K": list(k_range),
    "WCSS": wcss,
})

display(elbow_df.round(3))

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), wcss, marker="o", color="#028090")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method - Segmentasi Pelanggan")
plt.xticks(list(k_range))
plt.show()


**Interpretasi Elbow:** Dari grafik Elbow, penurunan WCSS terlihat besar pada K awal dan mulai melandai setelah sekitar **K = 3**. Karena dataset memang dibuat dengan tiga kelompok tersembunyi, maka K = 3 menjadi pilihan yang masuk akal untuk model K-Means.


## 5. Silhouette Score sebagai Pembanding

Silhouette Score digunakan sebagai pembanding yang lebih objektif. Nilainya berada pada rentang -1 sampai 1. Semakin mendekati 1, semakin baik kualitas pemisahan cluster.


In [ ]:
silhouette_rows = []

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, init="k-means++", n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_rows.append({
        "K": k,
        "Silhouette Score": silhouette_score(X_scaled, labels),
    })

silhouette_df = pd.DataFrame(silhouette_rows)
display(silhouette_df.round(3))

plt.figure(figsize=(8, 5))
sns.lineplot(data=silhouette_df, x="K", y="Silhouette Score", marker="o", color="#7b2cbf")
plt.title("Silhouette Score untuk Beberapa Nilai K")
plt.xticks(range(2, 8))
plt.show()


**Interpretasi Silhouette:** Jika K = 3 menghasilkan Silhouette Score yang tinggi dibanding kandidat lain, maka hasil ini mendukung keputusan dari Elbow Method. Artinya, pembagian pelanggan menjadi tiga segmen cukup jelas dan masuk akal.


## 6. Melatih Model K-Means

Model K-Means dilatih dengan `n_clusters=3` berdasarkan hasil Elbow Method dan pertimbangan Silhouette Score.


In [ ]:
k_optimal = 3

kmeans_model = KMeans(
    n_clusters=k_optimal,
    random_state=42,
    init="k-means++",
    n_init=10,
)

df["cluster"] = kmeans_model.fit_predict(X_scaled)

print(f"WCSS akhir       : {kmeans_model.inertia_:.3f}")
print(f"Silhouette Score : {silhouette_score(X_scaled, df['cluster']):.3f}")

cluster_summary = (
    df.groupby("cluster")[["pendapatan_tahunan", "skor_belanja", "usia"]]
    .agg(["mean", "min", "max"])
    .round(2)
)

display(cluster_summary)
display(df["cluster"].value_counts().sort_index().rename("jumlah_pelanggan"))


## 7. Visualisasi Hasil Clustering

Visualisasi berikut menampilkan hasil segmentasi pelanggan dengan warna berbeda untuk setiap cluster. Tanda X merah menunjukkan posisi centroid atau pusat cluster.


In [ ]:
centroids = scaler.inverse_transform(kmeans_model.cluster_centers_)

plt.figure(figsize=(8, 6))
plt.scatter(
    df["pendapatan_tahunan"],
    df["skor_belanja"],
    c=df["cluster"],
    cmap="viridis",
    alpha=0.75,
)
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c="red",
    marker="X",
    s=250,
    label="Centroid",
)
plt.xlabel("Pendapatan Tahunan (juta Rp)")
plt.ylabel("Skor Belanja")
plt.title("Segmentasi Pelanggan dengan K-Means")
plt.legend()
plt.show()

centroid_df = pd.DataFrame(centroids, columns=features)
centroid_df["cluster"] = range(k_optimal)
display(centroid_df[["cluster", "pendapatan_tahunan", "skor_belanja"]].round(2))


## 8. Interpretasi Segmen Pelanggan

Berdasarkan rata-rata pendapatan tahunan dan skor belanja, cluster dapat diberi interpretasi sebagai berikut. Nama segmen bisa berbeda tergantung nomor cluster yang terbentuk, tetapi maknanya tetap mengikuti karakteristik rata-ratanya.


In [ ]:
segment_summary = (
    df.groupby("cluster")
    .agg(
        jumlah_pelanggan=("cluster", "size"),
        rata_pendapatan=("pendapatan_tahunan", "mean"),
        rata_skor_belanja=("skor_belanja", "mean"),
        rata_usia=("usia", "mean"),
    )
    .round(2)
    .reset_index()
)

def segment_label(row):
    if row["rata_pendapatan"] < 50 and row["rata_skor_belanja"] < 40:
        return "Hemat"
    if row["rata_pendapatan"] > 90 and row["rata_skor_belanja"] > 70:
        return "Premium/Boros"
    return "Menengah"

segment_summary["interpretasi"] = segment_summary.apply(segment_label, axis=1)
display(segment_summary)


**Pembahasan:** Cluster dengan pendapatan dan skor belanja rendah dapat disebut segmen hemat. Cluster dengan pendapatan dan skor belanja sedang dapat disebut segmen menengah. Cluster dengan pendapatan tinggi dan skor belanja tinggi dapat disebut segmen premium atau pelanggan boros. Hasil ini dapat membantu bisnis menentukan strategi promosi yang berbeda untuk tiap kelompok pelanggan.


## 9. Hierarchical Clustering sebagai Pembanding

Hierarchical Clustering digunakan sebagai pembanding K-Means. Metode linkage yang dipakai adalah **Ward**, karena metode ini berusaha menghasilkan cluster yang relatif kompak.


In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(
    Z,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=45,
    leaf_font_size=10,
)
plt.title("Dendrogram - Segmentasi Pelanggan (Ward Linkage)")
plt.xlabel("Cluster Gabungan / Indeks Data")
plt.ylabel("Jarak Ward")
plt.axhline(y=8, color="red", linestyle="--", label="Contoh Garis Potong")
plt.legend()
plt.show()


In [ ]:
hier_model = AgglomerativeClustering(n_clusters=3, linkage="ward")
df["cluster_hierarchical"] = hier_model.fit_predict(X_scaled)

comparison = pd.crosstab(
    df["cluster"],
    df["cluster_hierarchical"],
    rownames=["K-Means"],
    colnames=["Hierarchical"],
)

display(comparison)

hier_summary = (
    df.groupby("cluster_hierarchical")[["pendapatan_tahunan", "skor_belanja"]]
    .mean()
    .round(2)
)
display(hier_summary)


**Interpretasi Dendrogram:** Dendrogram menunjukkan proses penggabungan data secara bertahap. Jika garis potong menghasilkan sekitar tiga kelompok besar, maka hasil Hierarchical Clustering konsisten dengan K optimal dari Elbow Method dan hasil K-Means.


## 10. Kesimpulan

Pada praktikum ini saya mempelajari bahwa clustering digunakan untuk mengelompokkan data tanpa label. Metode Elbow membantu menentukan jumlah cluster yang sesuai dengan melihat perubahan WCSS, sedangkan Silhouette Score membantu menilai kualitas pemisahan cluster. Pada dataset pelanggan sintetis ini, jumlah cluster yang paling masuk akal adalah **K = 3**, yaitu segmen hemat, menengah, dan premium. Hasil K-Means juga dapat dibandingkan dengan Hierarchical Clustering melalui dendrogram untuk melihat apakah pola segmentasi yang terbentuk konsisten.
